# TRIDENT: Full Pipeline Example

Runs the complete TRIDENT pipeline on `example.fasta` end-to-end. Mirrors
the steps available in the Streamlit app, with intermediate inspection of each
DataFrame and its associated cache parameters.

**Steps:** Load FASTA → MOL → TAX → GEO → EXTRA → HYPO → Results.

**Before running:**

- Set your contact email, required on every NCBI / GBIF / WoRMS / BOLD request.
- Fill in latitude / longitude / extents for the GEO step (study area).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import trident.clients as clients
import trident.pipelines as pipe
from trident import setup_logging
from trident.core import config
from trident.clients.ncbi import setup_ncbi_credentials

setup_logging("INFO")

### Configuration

Edit the constants in the next cell. Latitude / longitude / extents define the
study area for the GBIF geographic filter.

In [ ]:
# Contact email (REQUIRED). NCBI, GBIF, WoRMS and BOLD all expect one
# on every request. Read from a .env file at the repo root (gitignored),
# so your address never lands in the notebook. Create .env with:
#   CONTACT_EMAIL=you@domain.com
if not config.contact_email():
    raise RuntimeError(
        "CONTACT_EMAIL not set. Create a .env file at the repo root:\n"
        "    CONTACT_EMAIL=you@domain.com\n"
        "NCBI / GBIF / WoRMS / BOLD all require a contact email. See README."
    )
setup_ncbi_credentials()  # sets Entrez.email / NCBIWWW.email from .env

# Local SQLite cache for this run.
DB_PATH = "./results/example.db"

# Study area for the GEO (GBIF) step. Replace with your sampling site.
LATITUDE = 78.0  # TODO
LONGITUDE = 16.0  # TODO
EXTENTS = [500, "global"]  # km, plus 'global' as a baseline

FORCE_RERUN = False
FASTA_FILE = "example.fasta"

## Step 1: Load FASTA

Parse the input FASTA, build the sequences DataFrame, and cache it via the
fasta workflow wrapper so downstream caching keys can reference it.

In [ ]:
sequences = clients.fasta.load_fasta(FASTA_FILE)
print(f"Loaded {len(sequences)} sequences")

In [ ]:
sequences_df = clients.fasta.records_to_dataframe(sequences)
sequences_df

In [ ]:
# Preview the original FASTA string.
print(clients.fasta.records_to_fasta_string(sequences))

In [ ]:
# Full wrapper: caches the sequences_df keyed by file content.
sequences_df, _ = pipe.run_fasta_workflow(FASTA_FILE)
sequences_df

## Step 2: MOL (NCBI BLAST)

BLAST each sequence against NCBI, then filter the hit list by query coverage
and the barcoding-gap test. The retained species become the MOL output.

### Search

In [ ]:
# Keep batch_size small to stay below NCBI rate limits.
batch_size = 5
num_threads = 4
ev_exponent = 20  # E-value = 1e-20
max_hits = 500

In [ ]:
ncbi_sequences = pipe.prepare_ncbi_input(sequences_df)
ncbi_sequences

In [ ]:
ncbi_search_df, ncbi_search_params = pipe.run_ncbi_search(
    ncbi_sequences,
    batch_size=batch_size,
    num_threads=num_threads,
    ev_exponent=ev_exponent,
    max_hits=max_hits,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
ncbi_search_df.shape, list(ncbi_search_df.columns)

In [ ]:
ncbi_search_params

In [ ]:
ncbi_search_df.head()

In [ ]:
# Sanity-check the hit URL for the first row.
ncbi_search_df["hit_url"].iloc[0]

In [ ]:
# Per-sequence overview of the raw search:
# flags sequences where max_hits was saturated with a narrow identity range.
pipe.build_ncbi_search_overview(ncbi_search_df, sequences_df, max_hits=max_hits)

### Filtering

In [ ]:
query_cover = 90.0
gap_size = 2.0  # min identity drop defining the barcoding gap
method = "barcoding_gap"  # alternatives: "similarity"
gap_min_top = 97.0  # top-of-gap identity floor

In [ ]:
ncbi_filter_df, ncbi_filter_params = pipe.run_ncbi_filter(
    ncbi_search_df,
    query_cover=query_cover,
    gap_size=gap_size,
    method=method,
    gap_min_top=gap_min_top,
    search_params=ncbi_search_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
ncbi_filter_df.shape, list(ncbi_filter_df.columns)

In [ ]:
ncbi_filter_df.head()

In [ ]:
ncbi_filter_params

### Finalize MOL

In [ ]:
low_identity_threshold = 95.0  # below: low_identity_warning = True
enforce_threshold = False  # if True, drop those rows instead of warning

mol_df, mol_params = pipe.finalize_mol_results(
    ncbi_filter_df,
    threshold=low_identity_threshold,
    enforce_threshold=enforce_threshold,
    filter_params=ncbi_filter_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
mol_df.shape, list(mol_df.columns)

In [ ]:
mol_df.head()

In [ ]:
# Per-sequence MOL summary: kept species, identity range, warnings.
pipe.build_mol_summary(mol_df, sequences_df, ncbi_filter_df=ncbi_filter_df)

In [ ]:
# Per-sequence report including the barcoding-gap geometry, useful for
# manual inspection of a specific MOTU.
for seq_id in mol_df["seq_id"].unique():
    print(seq_id)
    print(pipe.build_mol_sequence_report(mol_df, seq_id, gap_size=gap_size))
    print("-" * 40)

## Step 3: TAX (WoRMS)

Expand every genus retained in MOL to its full marine species roster via
WoRMS. The expanded list becomes the candidate set for the GEO step.

In [ ]:
worms_input = pipe.prepare_worms_input(mol_df)
worms_input

In [ ]:
worms_search_df, worms_search_params = pipe.run_worms_search(
    worms_input,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
worms_search_df.shape, list(worms_search_df.columns)

In [ ]:
worms_search_df.head()

In [ ]:
worms_search_params

In [ ]:
worms_merge_df, worms_merge_params = pipe.run_worms_merge(
    worms_search_df,
    mol_df,
    mol_params=mol_params,
    worms_search_params=worms_search_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
worms_merge_df.shape, list(worms_merge_df.columns)

In [ ]:
worms_merge_df.head()

In [ ]:
# Species that WoRMS expansion brought in beyond what MOL already had.
pipe.build_tax_summary(worms_merge_df, mol_df=mol_df)

In [ ]:
tax_df, tax_params = pipe.finalize_tax_results(
    worms_merge_df,
    worms_merge_params=worms_merge_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
tax_df.shape

In [ ]:
tax_df.head()

## Step 4: GEO (GBIF)

Intersect candidate species with GBIF occurrence records inside the study
area. Species without enough occurrences in the chosen extent are dropped.

In [ ]:
# Study area defined at the top of the notebook
print(f"Study area: ({LATITUDE}, {LONGITUDE}) with extents {EXTENTS}")

In [ ]:
gbif_input = pipe.prepare_gbif_input(tax_df)
gbif_input

In [ ]:
gbif_search_df, gbif_search_params = pipe.run_gbif_search(
    gbif_input,
    latitude=LATITUDE,
    longitude=LONGITUDE,
    extents=EXTENTS,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
gbif_search_df.shape, list(gbif_search_df.columns)

In [ ]:
gbif_search_df.head()

In [ ]:
gbif_search_params

In [ ]:
gbif_merge_df, gbif_merge_params = pipe.run_gbif_merge(
    gbif_search_df,
    tax_df,
    tax_params=tax_params,
    gbif_search_params=gbif_search_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
gbif_merge_df.shape, list(gbif_merge_df.columns)

In [ ]:
gbif_merge_df.head()

In [ ]:
# Per-sequence GEO summary: how many candidate species have enough
# occurrences at each extent.
pipe.build_geo_summary(gbif_merge_df, min_occurrences=3)

### Filter

In [ ]:
extent = 500  # km, extent to count occurrences within
min_occurrences = 3  # minimum to keep a species

gbif_filter_df, gbif_filter_params = pipe.run_gbif_filter(
    gbif_merge_df,
    extent=extent,
    min_occurrences=min_occurrences,
    gbif_merge_params=gbif_merge_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
gbif_filter_df.shape, list(gbif_filter_df.columns)

In [ ]:
gbif_filter_df.head()

In [ ]:
gbif_filter_params

### Finalize GEO

In [ ]:
geo_df, geo_params = pipe.finalize_geo_results(
    gbif_filter_df,
    gbif_filter_params=gbif_filter_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
geo_df.shape

In [ ]:
geo_df.head()

In [ ]:
# Diagnostic: NCBI hits that survived MOL but were rejected at GEO
# (lost because GBIF could not place them in the study area).
pipe.get_ncbi_rejected_rows(gbif_filter_df, geo_df, ncbi_search_df).head()

## Step 5: EXTRA (BOLD)

For species without an NCBI reference for the query's marker, pull CO1 proxy
sequences from BOLD. These act as cross-marker stand-ins for the next step.

In [ ]:
bold_input, n_candidates = pipe.prepare_bold_input(
    geo_df, ncbi_search_df=ncbi_search_df
)
print(
    f"Querying BOLD for {len(bold_input)} species (out of {n_candidates} candidates)."
)
bold_input

In [ ]:
bold_search_df, bold_search_params = pipe.run_bold_search(
    bold_input,
    keep_only_COI5P=True,  # COI-5P sequences only
    keep_ncbi=False,  # exclude BOLD records that came from NCBI
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
bold_search_df.shape, list(bold_search_df.columns)

In [ ]:
bold_search_df.head()

In [ ]:
bold_search_params

In [ ]:
# Multiple BOLD records per species are common.
bold_search_df["scientificName"].value_counts()

### Filter (longest non-redundant)

In [ ]:
# Keep the longest non-redundant BOLD sequences per species.
similarity = 98.0  # sequences above this % identity are treated as redundant

bold_filter_df, bold_filter_params = pipe.run_bold_filter(
    bold_search_df,
    longest_n=None,  # None = keep all non-redundant; set an int to cap per species
    similarity=similarity,
    bold_search_params=bold_search_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
bold_filter_df.shape, list(bold_filter_df.columns)

In [ ]:
bold_filter_df.head()

### Merge & finalize

In [ ]:
bold_merge_df, bold_merge_params = pipe.run_bold_merge(
    bold_filter_df,
    geo_df,
    bold_filter_params=bold_filter_params,
    geo_params=geo_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
bold_merge_df.shape, list(bold_merge_df.columns)

In [ ]:
bold_merge_df.head()

In [ ]:
extra_df, extra_params = pipe.finalize_extra_results(
    bold_merge_df,
    bold_merge_params=bold_merge_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
extra_df.shape, list(extra_df.columns)

In [ ]:
extra_df.head()

In [ ]:
# Pair-level summary: which (seq_id, species) pairs got proxy sequences.
pipe.build_extra_summary(extra_df, geo_df, bold_input_species=bold_input)

In [ ]:
# Per-sequence rollup.
pipe.build_extra_seq_summary(
    pipe.build_extra_summary(extra_df, geo_df, bold_input_species=bold_input)
)

## Step 6: HYPO (proxy BLAST validation)

BLAST each BOLD proxy against NCBI. A species is confirmed indirectly when
its proxies align well with the query taxon, even when the species itself
has no NCBI reference for the query marker.

In [ ]:
hypo_input = pipe.prepare_hypo_input(extra_df, geo_df)
hypo_input

In [ ]:
hypo_search_df, hypo_search_params = pipe.run_hypo_search(
    hypo_input,
    batch_size=5,
    num_threads=10,
    ev_exponent=20,
    max_hits=500,
    identity_cutoff=90.0,
    ntop=3,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
hypo_search_df.shape, list(hypo_search_df.columns)

In [ ]:
hypo_search_df.head()

### Merge & filter

In [ ]:
hypo_merge_df, hypo_merge_params = pipe.run_hypo_merge(
    hypo_search_df,
    extra_df,
    hypo_search_params=hypo_search_params,
    extra_params=extra_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
hypo_merge_df.shape, list(hypo_merge_df.columns)

In [ ]:
hypo_merge_df.head()

In [ ]:
hypo_filter_df, hypo_filter_params = pipe.run_hypo_filter(
    hypo_merge_df,
    query_cover=50.0,
    identity=95.0,
    hypo_merge_params=hypo_merge_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
hypo_filter_df.shape, list(hypo_filter_df.columns)

In [ ]:
pipe.build_hypo_filter_summary(hypo_filter_df)

### Per-species NCBI check

In [ ]:
hypo_check_df, hypo_check_params = pipe.run_hypo_check(
    hypo_filter_df,
    ev_exponent=3,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
hypo_check_df.shape, list(hypo_check_df.columns)

In [ ]:
hypo_check_df.head()

In [ ]:
pipe.build_hypo_check_summary(hypo_check_df)

### Finalize HYPO

In [ ]:
hypo_df, hypo_params = pipe.finalize_hypo_results(
    hypo_filter_df,
    hypo_check_df,
    hypo_filter_params=hypo_filter_params,
    hypo_check_params=hypo_check_params,
    db_path=DB_PATH,
    force_rerun=FORCE_RERUN,
)
hypo_df.shape, list(hypo_df.columns)

In [ ]:
hypo_df.head()

## Step 7: Final Results

Merge the two validation paths (MOL + GEO, and HYPO) into a single species
ledger, annotate warnings, and export.

In [ ]:
results_df = pipe.build_results_df(
    sequences_df=sequences_df,
    geo_df=geo_df,
    mol_df=mol_df,
    hypo_df=hypo_df,
    ncbi_search_df=ncbi_search_df,
)
results_df.shape, list(results_df.columns)

In [ ]:
results_df

In [ ]:
# Flag rows whose identity is below the MOL low-identity threshold.
# results_df carries the MOL identity as 'ncbi_top_identity_percentage'
# (the default identity_col for both helpers below).
results_df = pipe.add_low_identity_warning(
    results_df,
    threshold=low_identity_threshold,
)

# Flag rows found in NCBI that would not have passed the MOL filter
# (identity at or below the strongest hit MOL rejected for that sequence).
results_df = pipe.add_below_mol(
    results_df,
    mol_df,
    ncbi_search_df,
)

results_df.head()

In [ ]:
# Diagnostic: which sequences were dropped, at which step.
all_seq_ids = sequences_df["seq_id"].tolist()
pipe.find_sequence_exclusion_step(
    all_seq_ids,
    ncbi_search_df,
    mol_df,
    tax_df,
    geo_df,
)

In [ ]:
# GBIF-occurrence export (formatted for submission back to GBIF).
gbif_export = pipe.build_gbif_export_df(results_df)
gbif_export

### Export

In [ ]:
import os

os.makedirs("results", exist_ok=True)

results_df.to_csv("results/example_results.csv", index=False)
gbif_export.to_csv("results/example_gbif_export.csv", index=False)

print("Wrote:")
print("  results/example_results.csv")
print("  results/example_gbif_export.csv")